# RxSafe
#### Prescription Safety Companion

Every day, millions of patients leave clinics holding prescriptions they don't fully understand, unaware of how new medicines interact with what they're already taking. RxSafe uses MedGemma to bridge this gap — not as a replacement for doctors, but as an intelligent safety net that catches what the system misses.

#### Architecture & Pipeline
It works using the MedGamma-4B-it model.

|Stage|Detail|
|:---|:---|
| **INPUT** | Patient uploads prescription image + fills profile (conditions, allergies, current medications) |
| ↓  |  |
| **VISION CALL** | MedGemma 4B multimodal extracts medicines, dosages, frequency, route, diagnosis as structured JSON |
| ↓  |  |
| **VALIDATE** | Python pre-validation: empty patient profile short-circuits to CLEAR, bypassing model to prevent hallucinated safety flags |
| ↓  |  |
| **TEXT CALL** | MedGemma 4B text: combined call generates patient education cards + drug-interaction / allergy safety check|
| ↓  |  |
| **OUTPUT** | UI renders: extracted medicines table, per-medicine education cards, color-coded safety flags (CRITICAL / CAUTION / CLEAR) |

In [1]:
! pip install --upgrade --quiet accelerate bitsandbytes transformers gradio 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 62.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.4 MB/s eta 0:00:00


In [2]:
# IMPORT LIBRARIES

#general
import numpy as np
import os
import requests
from io import BytesIO
import json
import re

#image
from PIL import Image, ImageFilter, ImageEnhance
from IPython.display import Image as IPImage, display

#model
import torch
from transformers import BitsAndBytesConfig, AutoProcessor, AutoModelForCausalLM

In [ ]:
# MODEL PARAMS
model_id = "google/medgemma-1.5-4b-it"

use_quantization = True #to reduce memory usage

# IMAGE PARAMS
image_path = "prescription-samples/0.jpg"

In [5]:
#To run in the notebook or launch the app
_launch_app = True

#### Load model

> MedGemma's weights are downloaded once during setup and stored on the clinic server. At runtime, the model runs entirely locally — the patient's phone connects over the clinic's WiFi, the prescription never leaves the building, and the clinic router doesn't need an internet connection at all.

In [6]:
model_kwargs = dict(
                    dtype=torch.bfloat16,
                    device_map="auto",
                    )

if use_quantization:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)

# One-time setup (with internet)
# huggingface-cli download google/medgemma-1.5-4b-it --local-dir ./models/medgemma
# Then Clinic runtime with no internet:

# Load model - For Demo
model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
processor = AutoProcessor.from_pretrained(model_id)
    
print("--> model loaded")

config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

--> model loaded


## Phase 1: Prescription Extraction

####  Helper Functions

In [7]:
def extract_prescription(image: Image.Image) -> dict:
    """
    Takes a PIL prescription image.
    Returns structured JSON with extracted medicines.
    """

    prompt = """You are a clinical AI assistant analyzing a prescription image.

                Extract ALL medicines prescribed. For each medicine return:
                - brand_name: name as written on prescription
                - generic_name: your best interpretation of the generic/chemical name
                - dosage: strength (e.g., 500mg)
                - frequency: how often (e.g., twice daily, TID)
                - duration: how long (e.g., 5 days, 1 month)
                - route: how taken (e.g., oral, topical) — if not mentioned write "oral" or try to extract from the type(cream, tablet, liquid etc.) of medicine
                
                Also extract:
                - doctor_notes: any special instructions written on prescription
                - diagnosis: condition mentioned if any
                
                Return ONLY a valid JSON object. No explanation, no markdown, no extra text.

                Example format:
                {
                  "medicines": [
                    {
                      "brand_name": "Dolo 650",
                      "generic_name": "Paracetamol 650mg",
                      "dosage": "650mg",
                      "frequency": "twice daily",
                      "duration": "5 days",
                      "route": "oral"
                    }
                  ],
                  "doctor_notes": "Take after food",
                  "diagnosis": "Viral fever"
                }"""

    messages = [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": prompt}]
                },
                {
                    "role": "user",
                    "content": [                       
                        {"type": "image", "image": image}
                    ]
                }
            ]
    
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, torch.bfloat16)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            temperature=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens
    input_len = inputs["input_ids"].shape[-1]
    decoded = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )

    return _parse_json_response(decoded)

In [8]:
def _parse_json_response(raw_text: str) -> dict:
    """
    Safely parse JSON from model output.
    Handles partial responses and nested JSON fragments.
    """
    # Try direct parse first
    try:
        return json.loads(raw_text.strip())
    except json.JSONDecodeError:
        pass

    # Find ALL JSON blocks in the response, pick the largest one
    # (the model sometimes returns a fragment instead of the full structure)
    matches = re.findall(r'\{[\s\S]*?\}(?=\s*[\{\[]|\s*$)', raw_text)
    
    if not matches:
        match = re.search(r'\{[\s\S]*\}', raw_text)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    else:
        # Try each match, largest first
        for candidate in sorted(matches, key=len, reverse=True):
            try:
                return json.loads(candidate)
            except json.JSONDecodeError:
                continue

    print(f"⚠️ Could not parse JSON. Raw output:\n{raw_text}")
    return {
        "medicines": [],
        "raw_output": raw_text,
        "parse_error": True
    }

In [9]:
def preprocess_prescription(image: Image.Image) -> Image.Image:
    # Convert to grayscale
    image = image.convert("L")
    
    # Increase contrast
    enhancer = ImageEnhance.Contrast(image)
    image = enhancer.enhance(2.5)
    
    # Sharpen
    image = image.filter(ImageFilter.SHARPEN)
    
    # Back to RGB for the model
    return image.convert("RGB")

#### Phase 1: Dry Run

In [10]:
def run_prescription_extraction(image_path):
    # open image
    image = Image.open(image_path).convert("RGB")
    display(IPImage(filename=image_path, height=500, width=300))

    #enhance image
    #image = preprocess_prescription(image)
    
    # Run extraction
    print("Running prescription extraction...")
    result = extract_prescription(image)
    
    # Pretty print result
    print("\n✅ Extracted Prescription Data:")
    print(json.dumps(result, indent=2))
    
    # Quick validation
    if result.get("parse_error"):
        print("\n⚠️ WARNING: JSON parsing failed. Check raw_output above.")
    elif not result.get("medicines"):
        print("\n⚠️ WARNING: No medicines extracted. Try a clearer image.")
    else:
        print(f"\n✅ Successfully extracted {len(result['medicines'])} medicine(s)")
        for med in result["medicines"]:
            print(f"  → {med.get('brand_name')} | {med.get('generic_name')} | {med.get('dosage')} | {med.get('frequency')}")

    return result


if not _launch_app:
    # run prescription extraction pipeline
    result = run_prescription_extraction(image_path)

## Phase 2: Patient Safety Check

#### Patient sample profile

In [11]:
sample_patient_profile = {
    "age": 68,
    "gender": "Female",
    "weight_kg": 59,
    "conditions": ["Type 2 Diabetes", "Hypertension"],
    "allergies": ["Penicillin"],
    "current_medications": [
        {"name": "Metformin", "dosage": "500mg", "frequency": "twice daily"},
        {"name": "Amlodipine", "dosage": "5mg", "frequency": "once daily"}
    ]
}

#### Safety Check Helper Function

In [12]:
def analyze_medicines(enriched: dict, patient_profile: dict) -> dict:
    """
    Returns both education content and safety analysis in one response.
    """

    medicines_text = ""
    for med in enriched.get("medicines", []):
        medicines_text += f"""
  - {med['brand_name']} ({med['generic_name']})
    Dosage: {med['dosage']} | Frequency: {med['frequency']} | Duration: {med['duration']} | Route: {med['route']}"""
        
    conditions_text  = ", ".join(patient_profile.get("conditions",  [])) or "None reported"
    allergies_text   = ", ".join(patient_profile.get("allergies",   [])) or "None reported"
    current_meds_text = "\n".join([
        f"  - {m['name']} {m.get('dosage','')} {m.get('frequency','')}"
        for m in patient_profile.get("current_medications", [])
    ]) or "  None reported"

    prompt = f"""You are a clinical pharmacist AI. Analyze this prescription and return a single 
            comprehensive JSON response covering both patient education and safety analysis.
            
            PATIENT PROFILE:
              Age: {patient_profile.get('age', 'Unknown')} | Gender: {patient_profile.get('gender', 'Unknown')} | Weight: {patient_profile.get('weight_kg', 'Unknown')} kg
              Known Conditions: {conditions_text}
              Known Allergies: {allergies_text}
              Current Medications:
            {current_meds_text}
            
            PRESCRIPTION:
              Diagnosis: {enriched.get('diagnosis', 'Not specified')}
              Doctor Notes: {enriched.get('doctor_notes', 'None')}
              Medicines:
            {medicines_text}
            
            Return ONLY a valid JSON object with this exact structure:

            {{
              "education": {{
                "medicines": [
                  {{
                    "brand_name": "name as written",
                    "generic_name": "generic name",
                    "plain_name": "simple name e.g. antibiotic, painkiller",
                    "what_it_is": "1-2 sentences in simple words",
                    "why_prescribed": "why doctor likely prescribed this",
                    "how_to_take": "clear instructions — timing, food, water",
                    "common_side_effects": ["effect 1", "effect 2", "effect 3"],
                    "what_to_avoid": ["avoid 1", "avoid 2"],
                    "tips": "1-2 practical tips",
                    "pharmacist_guidance": {{
                      "out_of_stock_advice": "what to tell pharmacist if unavailable — include generic name, drug class, condition treated",
                      "questions_to_ask": ["question 1", "question 2"],
                      "red_flags": "when to call doctor instead of accepting substitution"
                    }}
                  }}
                ],
                "general_advice": "1-2 sentences of overall advice for the prescription course"
              }},

              "safety": {{
                "overall_safety": "CRITICAL | CAUTION | CLEAR",
                "flags": [
                  {{
                    "type": "ALLERGY | DRUG_INTERACTION | CONTRAINDICATION | DOSAGE_CONCERN | DUPLICATE_THERAPY",
                    "severity": "HIGH | MEDIUM | LOW",
                    "medicine_involved": "medicine name",
                    "explanation": "plain English explanation of the risk",
                    "recommendation": "what patient should do"
                  }}
                ],
                "patient_friendly_summary": "2-3 sentence plain English summary",
                "safe_to_take": true,
                "consult_doctor_urgently": false
              }}
            }}
            
            STRICT SAFETY RULES — violations will harm patients:
            1. ALLERGY: ONLY flag if patient EXPLICITLY reported an allergy matching a prescribed medicine
            2. DOSAGE: ONLY flag if dosage clearly exceeds published standard guidelines
            3. CONTRAINDICATION: ONLY flag if patient reported a condition that specifically contraindicates the medicine
            4. DUPLICATE_THERAPY: ONLY flag if patient is actively taking another medicine with same mechanism
            5. Missing/unspecified data = neutral. NEVER invent risks to fill gaps
            6. If no real flags exist: overall_safety = "CLEAR", flags = [], safe_to_take = true
            
            Return ONLY the JSON object. No explanation, no markdown, no extra text."""

    messages = [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, torch.bfloat16)

    #token optimization
    # Scale max tokens by number of medicines
    num_medicines = len(enriched.get("medicines", []))
    max_tokens = min(800 + (num_medicines * 600), 3000)
    
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,    
            do_sample=False,
            temperature=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[-1]
    decoded = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    return _parse_json_response(decoded)


def split_analysis(raw: dict) -> tuple[dict, dict]:
    """
    Splits the combined response into education and safety dicts.
    Handles partial responses gracefully.
    """
    education = raw.get("education", {})
    safety    = raw.get("safety",    {})

    # If model returned flat structure, try to reconstruct
    if not education and not safety:
        if "medicines" in raw:
            education = raw  # treat whole response as education
        if "overall_safety" in raw:
            safety = raw     # treat whole response as safety

    # Apply safety reconstruction if needed
    if safety:
        safety = reconstruct_safety_response(safety)
    else:
        safety = {
            "overall_safety": "UNKNOWN",
            "flags": [],
            "patient_friendly_summary": "Safety analysis could not be completed.",
            "safe_to_take": False,
            "consult_doctor_urgently": False,
            "parse_error": True
        }

    return education, safety

In [13]:
def reconstruct_safety_response(raw: dict) -> dict:
    """
    If model returns a partial response (just a flag object),
    reconstruct the full expected safety structure around it.
    """
    # Check if this looks like a full valid response already
    if "overall_safety" in raw and "flags" in raw:
        return raw  # already correct structure, return as-is

    # Check if it looks like a single flag object
    if "type" in raw and "severity" in raw and "explanation" in raw:
        severity = raw.get("severity", "LOW")
        overall = "CRITICAL" if severity == "HIGH" else "CAUTION"
        return {
            "overall_safety": overall,
            "flags": [raw],
            "patient_friendly_summary": raw.get("explanation", ""),
            "safe_to_take": severity == "HIGH",
            "consult_doctor_urgently": severity == "HIGH",
            "_reconstructed": True  # flag so we know this was patched
        }

    # Unknown partial structure — return as error
    return {
        "overall_safety": "UNKNOWN",
        "flags": [],
        "patient_friendly_summary": "Could not parse safety response.",
        "safe_to_take": False,
        "consult_doctor_urgently": False,
        "parse_error": True,
        "raw_output": raw
    }

In [14]:
def validate_safety_inputs(enriched: dict, patient_profile: dict) -> dict | None:
    """
    Returns a CLEAR result immediately if there's genuinely nothing to check.
    Returns None if we should proceed with the full model safety check.
    """
    has_allergies    = bool(patient_profile.get("allergies"))
    has_conditions   = bool(patient_profile.get("conditions"))
    has_current_meds = bool(patient_profile.get("current_medications") and 
                           any(m.get("name") for m in patient_profile["current_medications"]))

    # If patient profile is completely empty, skip the model — nothing to cross-check
    if not has_allergies and not has_conditions and not has_current_meds:
        med_names = [m.get("brand_name", "") for m in enriched.get("medicines", [])]
        return {
            "overall_safety": "CLEAR",
            "flags": [],
            "patient_friendly_summary": (
                f"Your prescription contains {', '.join(med_names)}. "
                f"No safety concerns were identified based on the information provided. "
                f"For a more detailed safety check, add your known allergies, "
                f"medical conditions, and current medications to your profile."
            ),
            "safe_to_take": True,
            "consult_doctor_urgently": False,
            "_skipped_reason": "Insufficient patient profile to perform meaningful safety check"
        }

    return None  # proceed with full model check

#### Phase 2: Dry Run

In [15]:
def run_safety_check_pipeline(result, patient_profile):
    print("Running safety check...")

    # Short-circuit safety if profile is empty — still run education
    skip_safety = validate_safety_inputs(result, patient_profile) 

    raw_analysis = analyze_medicines(result, patient_profile)
    education, safety = split_analysis(raw_analysis)

    
    if skip_safety:
        safety = skip_safety
        print("⚡ Safety short-circuited — empty patient profile detected")

    # ── Text summary ──────────────────────────────────────────────────────
    print(f"\n✅ Education — medicines explained : {len(education.get('medicines', []))}")
    print(f"✅ Education — keys               : {list(education.keys())}")
    print(f"✅ Safety    — overall status      : {safety.get('overall_safety')}")
    print(f"✅ Safety    — flags found         : {len(safety.get('flags', []))}")
    print(f"✅ Safety    — safe to take        : {safety.get('safe_to_take')}")
    print(f"✅ Safety    — consult urgently    : {safety.get('consult_doctor_urgently')}")

    if safety.get("flags"):
        print("\n  Flags detected:")
        for flag in safety.get("flags", []):
            print(f"    [{flag.get('severity')}] {flag.get('type')} — {flag.get('medicine_involved')}")
            explanation = flag.get('explanation', '')
            print(f"    {explanation[:120]}{'...' if len(explanation) > 120 else ''}")
    else:
        print("\n  No flags raised.")



if not _launch_app:
    # Run the full pipeline
    run_safety_check_pipeline(result, sample_patient_profile)

## The App

In [16]:
import gradio as gr

#### UI Helper Functions

In [17]:
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&family=Syne:wght@600;700;800&display=swap');

/* ── Base ── */
* { box-sizing: border-box; }

body, .gradio-container {
    background: #0a0f1e !important;
    font-family: 'DM Sans', sans-serif !important;
    color: #e2e8f0 !important;
    min-height: 100vh;
}

.gradio-container {
    background: radial-gradient(ellipse at 20% 20%, #0d1f3c 0%, #0a0f1e 60%) !important;
    max-width: 1280px !important;
}

/* ── Header ── */
.rx-header {
    text-align: center;
    padding: 48px 24px 32px;
    position: relative;
}
.rx-header::after {
    content: '';
    display: block;
    width: 80px;
    height: 3px;
    background: linear-gradient(90deg, #00d4aa, #0099ff);
    margin: 20px auto 0;
    border-radius: 2px;
}
.rx-logo {
    font-family: 'Syne', sans-serif;
    font-size: 2.8em;
    font-weight: 800;
    background: linear-gradient(135deg, #00d4aa 0%, #0099ff 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    letter-spacing: -1px;
    margin: 0;
}
.rx-tagline {
    color: #64748b;
    font-size: 0.95em;
    font-weight: 300;
    margin: 8px 0 0;
    letter-spacing: 0.5px;
}
.rx-disclaimer {
    display: inline-block;
    margin-top: 14px;
    background: rgba(255,170,0,0.08);
    border: 1px solid rgba(255,170,0,0.2);
    color: #ffaa00;
    font-size: 0.78em;
    padding: 6px 16px;
    border-radius: 20px;
    letter-spacing: 0.3px;
}

/* ── Panel title ── */
.panel-title {
    font-family: 'Syne', sans-serif;
    font-size: 0.75em;
    font-weight: 700;
    letter-spacing: 2px;
    text-transform: uppercase;
    color: #00d4aa;
    margin: 0 0 16px;
    display: flex;
    align-items: center;
    gap: 8px;
}

/* ── Divider ── */
.section-divider {
    border: none;
    border-top: 1px solid rgba(255,255,255,0.06);
    margin: 20px 0;
}

/* ── Gradio component overrides ── */
.gr-block, .gr-form, .gr-box {
    background: transparent !important;
    border: none !important;
}
label, .gr-input-label {
    color: #94a3b8 !important;
    font-size: 0.82em !important;
    font-weight: 500 !important;
    letter-spacing: 0.5px !important;
    text-transform: uppercase !important;
}
input[type="number"], textarea, select, .gr-input, .gr-textarea {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
    border-radius: 10px !important;
    color: #e2e8f0 !important;
    font-family: 'DM Sans', sans-serif !important;
    font-size: 0.95em !important;
    transition: border-color 0.2s ease !important;
}
input[type="number"]:focus, textarea:focus {
    border-color: #00d4aa !important;
    outline: none !important;
    box-shadow: 0 0 0 3px rgba(0,212,170,0.1) !important;
}

/* ── Upload box ── */
.upload-box .gr-image, .upload-box {
    border: 2px dashed rgba(0,212,170,0.3) !important;
    border-radius: 14px !important;
    background: rgba(0,212,170,0.03) !important;
    transition: border-color 0.2s ease !important;
}
.upload-box:hover {
    border-color: rgba(0,212,170,0.6) !important;
}

/* ── Buttons ── */
.btn-primary {
    background: linear-gradient(135deg, #00d4aa 0%, #0099ff 100%) !important;
    border: none !important;
    border-radius: 12px !important;
    color: #0a0f1e !important;
    font-family: 'Syne', sans-serif !important;
    font-size: 1em !important;
    font-weight: 700 !important;
    letter-spacing: 0.5px !important;
    padding: 14px 24px !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;
    width: 100% !important;
}
.btn-primary:hover {
    transform: translateY(-1px) !important;
    box-shadow: 0 8px 24px rgba(0,212,170,0.3) !important;
}
.btn-secondary {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
    border-radius: 12px !important;
    color: #94a3b8 !important;
    font-family: 'DM Sans', sans-serif !important;
    font-size: 0.88em !important;
    padding: 10px 24px !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;
    width: 100% !important;
}
.btn-secondary:hover {
    background: rgba(255,255,255,0.08) !important;
    color: #e2e8f0 !important;
}

/* ── Dropdown ── */
.gr-dropdown select {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
    border-radius: 10px !important;
    color: #e2e8f0 !important;
}

/* ── Tabs container ── */
#output-tabs {
    background: transparent !important;
    border: none !important;
}

/* ── Tab navigation bar ── */
#output-tabs .tab-nav {
    background: rgba(255,255,255,0.03) !important;
    border: 1px solid rgba(255,255,255,0.07) !important;
    border-radius: 12px !important;
    padding: 4px !important;
    gap: 4px !important;
    margin-bottom: 16px !important;
}

/* ── Individual tab buttons ── */
#output-tabs .tab-nav button {
    background: transparent !important;
    border: none !important;
    border-radius: 8px !important;
    color: #64748b !important;
    font-family: 'DM Sans', sans-serif !important;
    font-size: 0.85em !important;
    font-weight: 500 !important;
    padding: 8px 16px !important;
    transition: all 0.2s ease !important;
    flex: 1 !important;
    text-align: center !important;
}
#output-tabs .tab-nav button:hover {
    color: #e2e8f0 !important;
    background: rgba(255,255,255,0.05) !important;
}
#output-tabs .tab-nav button.selected {
    background: linear-gradient(135deg, rgba(0,212,170,0.15), rgba(0,153,255,0.15)) !important;
    color: #00d4aa !important;
    font-weight: 600 !important;
    border: 1px solid rgba(0,212,170,0.2) !important;
}

/* ── Tab content area ── */
#output-tabs > .tabitem {
    background: transparent !important;
    border: none !important;
    padding: 0 !important;
}

/* ── Medicines output — scrollable ── */
#medicines-out {
    background: transparent !important;
    border: none !important;
    padding: 0 4px 0 0 !important;
    max-height: 600px !important;
    min-height: 60px !important;
    overflow-y: auto !important;
    overflow-x: hidden !important;
    display: block !important;
    position: relative !important;
    scrollbar-width: thin;
    scrollbar-color: #00d4aa44 transparent;
}
#medicines-out::-webkit-scrollbar { width: 5px; }
#medicines-out::-webkit-scrollbar-track { background: transparent; }
#medicines-out::-webkit-scrollbar-thumb {
    background: #00d4aa44;
    border-radius: 10px;
}
#medicines-out::-webkit-scrollbar-thumb:hover { background: #00d4aa99; }

/* ── Education output — scrollable ── */
#education-out {
    background: transparent !important;
    border: none !important;
    padding: 0 4px 0 0 !important;
    max-height: 600px !important;
    min-height: 60px !important;
    overflow-y: auto !important;
    overflow-x: hidden !important;
    display: block !important;
    position: relative !important;
    scrollbar-width: thin;
    scrollbar-color: #00d4aa44 transparent;
}
#education-out::-webkit-scrollbar { width: 5px; }
#education-out::-webkit-scrollbar-track { background: transparent; }
#education-out::-webkit-scrollbar-thumb {
    background: #00d4aa44;
    border-radius: 10px;
}
#education-out::-webkit-scrollbar-thumb:hover { background: #00d4aa99; }

/* ── Safety output — scrollable ── */
#safety-out {
    background: transparent !important;
    border: none !important;
    padding: 0 4px 0 0 !important;
    max-height: 600px !important;
    min-height: 60px !important;
    overflow-y: auto !important;
    overflow-x: hidden !important;
    display: block !important;
    position: relative !important;
    scrollbar-width: thin;
    scrollbar-color: #0099ff44 transparent;
}
#safety-out::-webkit-scrollbar { width: 5px; }
#safety-out::-webkit-scrollbar-track { background: transparent; }
#safety-out::-webkit-scrollbar-thumb {
    background: #0099ff44;
    border-radius: 10px;
}
#safety-out::-webkit-scrollbar-thumb:hover { background: #0099ff99; }

@keyframes spin {
    0%   { transform: rotate(0deg); }
    100% { transform: rotate(360deg); }
}
"""

In [18]:
# ── HTML Renderers ─────────────────────────────────────────────────────────────

def medicines_to_html(extracted: dict) -> str:
    medicines = extracted.get("medicines", [])
    if not medicines:
        return """
        <div style='text-align:center; padding:40px; color:#475569; font-family:DM Sans,sans-serif;'>
            No medicines extracted yet.
        </div>"""

    diagnosis = extracted.get("diagnosis", "")
    notes     = extracted.get("doctor_notes", "")

    meta = ""
    if diagnosis:
        meta += f"<span style='background:rgba(0,212,170,0.1); color:#00d4aa; padding:4px 12px; border-radius:20px; font-size:0.82em; margin-right:8px;'>📋 {diagnosis}</span>"
    if notes:
        meta += f"<span style='background:rgba(0,153,255,0.1); color:#0099ff; padding:4px 12px; border-radius:20px; font-size:0.82em;'>📝 {notes}</span>"

    rows = ""
    for m in medicines:
        rows += f"""
        <tr>
          <td style='padding:14px 16px; border-bottom:1px solid rgba(255,255,255,0.05);'>
            <div style='font-weight:600; color:#e2e8f0; font-size:0.97em;'>{m.get("brand_name","")}</div>
            <div style='color:#64748b; font-size:0.82em; margin-top:2px;'>{m.get("generic_name","")}</div>
          </td>
          <td style='padding:14px 16px; border-bottom:1px solid rgba(255,255,255,0.05); color:#00d4aa; font-weight:500;'>{m.get("dosage","—")}</td>
          <td style='padding:14px 16px; border-bottom:1px solid rgba(255,255,255,0.05); color:#94a3b8;'>{m.get("frequency","—")}</td>
          <td style='padding:14px 16px; border-bottom:1px solid rgba(255,255,255,0.05); color:#94a3b8;'>{m.get("duration","—")}</td>
          <td style='padding:14px 16px; border-bottom:1px solid rgba(255,255,255,0.05);'>
            <span style='background:rgba(255,255,255,0.06); color:#94a3b8; padding:3px 10px; border-radius:20px; font-size:0.8em;'>{m.get("route","oral")}</span>
          </td>
        </tr>"""

    return f"""
    <div style='font-family:DM Sans,sans-serif; color:#e2e8f0;'>
      {f'<div style="margin-bottom:16px;">{meta}</div>' if meta else ''}
      <div style='background:rgba(255,255,255,0.03); border:1px solid rgba(255,255,255,0.07); border-radius:14px; overflow:hidden;'>
        <table style='width:100%; border-collapse:collapse;'>
          <thead>
            <tr style='background:rgba(0,212,170,0.06);'>
              <th style='padding:12px 16px; text-align:left; font-size:0.75em; letter-spacing:1.5px; text-transform:uppercase; color:#00d4aa; font-weight:600;'>Medicine</th>
              <th style='padding:12px 16px; text-align:left; font-size:0.75em; letter-spacing:1.5px; text-transform:uppercase; color:#00d4aa; font-weight:600;'>Dosage</th>
              <th style='padding:12px 16px; text-align:left; font-size:0.75em; letter-spacing:1.5px; text-transform:uppercase; color:#00d4aa; font-weight:600;'>Frequency</th>
              <th style='padding:12px 16px; text-align:left; font-size:0.75em; letter-spacing:1.5px; text-transform:uppercase; color:#00d4aa; font-weight:600;'>Duration</th>
              <th style='padding:12px 16px; text-align:left; font-size:0.75em; letter-spacing:1.5px; text-transform:uppercase; color:#00d4aa; font-weight:600;'>Route</th>
            </tr>
          </thead>
          <tbody>{rows}</tbody>
        </table>
      </div>
    </div>"""

def safety_to_html(safety: dict) -> str:
    if safety.get("parse_error"):
        return """
        <div style='background:rgba(255,68,68,0.08); border:1px solid rgba(255,68,68,0.2); 
                    border-radius:14px; padding:20px; color:#ff4444; font-family:DM Sans,sans-serif;'>
            ❌ Could not analyze prescription. Please try again with a clearer image.
        </div>"""

    status = safety.get("overall_safety", "UNKNOWN")
    theme = {
        "CRITICAL": {"bg": "rgba(255,68,68,0.08)",  "border": "rgba(255,68,68,0.3)",  "accent": "#ff4444", "emoji": "🔴"},
        "CAUTION":  {"bg": "rgba(255,170,0,0.08)",  "border": "rgba(255,170,0,0.3)",  "accent": "#ffaa00", "emoji": "🟡"},
        "CLEAR":    {"bg": "rgba(0,204,102,0.08)",  "border": "rgba(0,204,102,0.3)",  "accent": "#00cc66", "emoji": "🟢"},
        "UNKNOWN":  {"bg": "rgba(100,116,139,0.08)","border": "rgba(100,116,139,0.3)","accent": "#64748b", "emoji": "⚪"},
    }.get(status, {"bg": "rgba(100,116,139,0.08)", "border": "rgba(100,116,139,0.3)", "accent": "#64748b", "emoji": "⚪"})

    t = theme

    safe_badge = (
        f"<span style='background:rgba(0,204,102,0.15); color:#00cc66; padding:4px 14px; border-radius:20px; font-size:0.82em;'>✅ Safe to take</span>"
        if safety.get("safe_to_take") else
        f"<span style='background:rgba(255,68,68,0.15); color:#ff4444; padding:4px 14px; border-radius:20px; font-size:0.82em;'>❌ Do NOT take</span>"
    )
    urgent_badge = (
        f"<span style='background:rgba(255,68,68,0.15); color:#ff4444; padding:4px 14px; border-radius:20px; font-size:0.82em; margin-left:8px;'>⚠️ See doctor urgently</span>"
        if safety.get("consult_doctor_urgently") else ""
    )

    # Status banner
    html = f"""
    <div style='font-family:DM Sans,sans-serif; color:#e2e8f0;'>
      <div style='background:{t["bg"]}; border:1px solid {t["border"]}; 
                  border-radius:16px; padding:22px 24px; margin-bottom:16px;'>
        <div style='display:flex; align-items:center; justify-content:space-between; flex-wrap:wrap; gap:10px;'>
          <div>
            <div style='font-family:Syne,sans-serif; font-size:1.4em; font-weight:800; color:{t["accent"]};'>
              {t["emoji"]} {status}
            </div>
            <div style='color:#64748b; font-size:0.82em; margin-top:4px; letter-spacing:0.3px;'>
              Safety Assessment Complete
            </div>
          </div>
          <div style='display:flex; gap:8px; flex-wrap:wrap;'>
            {safe_badge}{urgent_badge}
          </div>
        </div>
      </div>
    """

    # Summary box
    summary = safety.get("patient_friendly_summary", "")
    if summary:
        html += f"""
      <div style='background:rgba(255,255,255,0.03); border:1px solid rgba(255,255,255,0.07);
                  border-radius:14px; padding:18px 20px; margin-bottom:16px;'>
        <div style='font-size:0.72em; letter-spacing:2px; text-transform:uppercase; 
                    color:#00d4aa; font-weight:600; margin-bottom:10px;'>📋 Patient Summary</div>
        <p style='margin:0; color:#cbd5e1; line-height:1.7; font-size:0.95em;'>{summary}</p>
      </div>"""

    # Flags
    flags = safety.get("flags", [])
    if not flags:
        html += """
      <div style='background:rgba(0,204,102,0.06); border:1px solid rgba(0,204,102,0.15);
                  border-radius:14px; padding:18px 20px; text-align:center; color:#00cc66;'>
        ✅ No safety concerns identified.
      </div>"""
    else:
        sev_theme = {
            "HIGH":   {"bg": "rgba(255,68,68,0.06)",  "border": "rgba(255,68,68,0.2)",  "color": "#ff4444",  "badge": "rgba(255,68,68,0.15)",  "emoji": "🔴"},
            "MEDIUM": {"bg": "rgba(255,170,0,0.06)",  "border": "rgba(255,170,0,0.2)",  "color": "#ffaa00",  "badge": "rgba(255,170,0,0.15)",  "emoji": "🟡"},
            "LOW":    {"bg": "rgba(0,153,255,0.06)",  "border": "rgba(0,153,255,0.2)",  "color": "#0099ff",  "badge": "rgba(0,153,255,0.15)",  "emoji": "🔵"},
        }

        html += f"""
      <div style='font-size:0.72em; letter-spacing:2px; text-transform:uppercase;
                  color:#94a3b8; font-weight:600; margin-bottom:12px;'>
        ⚠️ {len(flags)} Safety Flag{'s' if len(flags) > 1 else ''} Found
      </div>"""

        for flag in flags:
            sev = flag.get("severity", "LOW")
            st = sev_theme.get(sev, sev_theme["LOW"])
            flag_type = flag.get("type", "").replace("_", " ")

            html += f"""
      <div style='background:{st["bg"]}; border:1px solid {st["border"]};
                  border-radius:14px; padding:18px 20px; margin-bottom:12px;'>
        <div style='display:flex; align-items:center; gap:10px; margin-bottom:12px;'>
          <span style='background:{st["badge"]}; color:{st["color"]}; padding:4px 12px;
                       border-radius:20px; font-size:0.78em; font-weight:600; letter-spacing:0.5px;'>
            {st["emoji"]} {sev}
          </span>
          <span style='color:{st["color"]}; font-size:0.82em; font-weight:600; 
                       letter-spacing:1px; text-transform:uppercase;'>{flag_type}</span>
          <span style='color:#475569; font-size:0.85em; margin-left:auto;'>{flag.get("medicine_involved","")}</span>
        </div>
        <p style='margin:0 0 12px; color:#cbd5e1; line-height:1.65; font-size:0.93em;'>
          {flag.get("explanation","")}
        </p>
        <div style='background:rgba(255,255,255,0.04); border-radius:10px; padding:12px 14px;
                    border-left:3px solid {st["color"]};'>
          <span style='color:#94a3b8; font-size:0.78em; text-transform:uppercase; 
                       letter-spacing:1px; font-weight:600;'>Recommended Action</span>
          <p style='margin:6px 0 0; color:#e2e8f0; font-size:0.92em; line-height:1.6;'>
            {flag.get("recommendation","")}
          </p>
        </div>
      </div>"""

    html += "</div>"
    return html

In [19]:
def education_to_html(education: dict) -> str:
    if not education or education.get("parse_error"):
        return """
        <div style='text-align:center; padding:40px; color:#475569; font-family:DM Sans,sans-serif;'>
            Could not generate medicine information. Please try again.
        </div>"""

    medicines = education.get("medicines", [])
    general_advice = education.get("general_advice", "")
    html = "<div style='font-family:DM Sans,sans-serif; color:#e2e8f0;'>"

    for med in medicines:
        side_effects = "".join(
            f"<span style='background:rgba(255,170,0,0.1); color:#ffaa00; padding:3px 10px; "
            f"border-radius:20px; font-size:0.8em; margin:3px 3px 3px 0; display:inline-block;'>"
            f"{se}</span>"
            for se in med.get("common_side_effects", [])
        )

        avoid_items = "".join(
            f"<li style='color:#94a3b8; font-size:0.9em; margin-bottom:4px;'>🚫 {a}</li>"
            for a in med.get("what_to_avoid", [])
        )

        # OTC alternatives section
        alts = [a for a in med.get("otc_alternatives", [])
                if a.get("fda_otc_approved") and not a.get("requires_doctor")]

        alts_html = ""
        if alts:
            alt_cards = ""
            for alt in alts:
                alt_cards += f"""
                <div style='background:rgba(0,153,255,0.06); border:1px solid rgba(0,153,255,0.15);
                            border-radius:10px; padding:12px 14px; margin-bottom:8px;'>
                  <div style='display:flex; align-items:center; gap:8px; margin-bottom:6px;'>
                    <b style='color:#0099ff;'>{alt.get("name","")}</b>
                    <span style='background:rgba(0,204,102,0.15); color:#00cc66; padding:2px 8px;
                                 border-radius:20px; font-size:0.75em;'>✅ FDA OTC Approved</span>
                    <span style='background:rgba(0,204,102,0.1); color:#00cc66; padding:2px 8px;
                                 border-radius:20px; font-size:0.75em;'>No prescription needed</span>
                  </div>
                  <div style='color:#94a3b8; font-size:0.85em; margin-bottom:4px;'>
                    {alt.get("why_suitable","")}
                  </div>
                  {f'<div style="color:#ffaa00; font-size:0.82em; margin-top:4px;">⚠️ {alt.get("note","")}</div>' 
                   if alt.get("note") else ""}
                </div>"""

            alts_html = f"""
            <div style='margin-top:16px;'>
              <div style='font-size:0.72em; letter-spacing:2px; text-transform:uppercase;
                          color:#0099ff; font-weight:600; margin-bottom:10px;'>
                🔄 If Out of Stock — FDA Approved OTC Alternatives
              </div>
              {alt_cards}
              <p style='color:#475569; font-size:0.78em; margin-top:8px;'>
                ℹ️ These alternatives treat the same condition and are available without a prescription. 
                Still inform your doctor if you switch.
              </p>
            </div>"""
        else:
            alts_html = """
            <div style='margin-top:16px; padding:10px 14px; background:rgba(255,255,255,0.03);
                        border-radius:10px; color:#475569; font-size:0.85em;'>
                🔄 No OTC alternatives available for this medicine — 
                consult your pharmacist or doctor if it is out of stock.
            </div>"""

        html += f"""
        <div style='background:rgba(255,255,255,0.03); border:1px solid rgba(255,255,255,0.07);
                    border-radius:16px; padding:22px 24px; margin-bottom:20px;'>

          <!-- Medicine Header -->
          <div style='display:flex; align-items:flex-start; justify-content:space-between; 
                      flex-wrap:wrap; gap:10px; margin-bottom:18px;'>
            <div>
              <h3 style='margin:0; font-family:Syne,sans-serif; font-size:1.3em; 
                          font-weight:800; color:#e2e8f0;'>{med.get("brand_name","")}</h3>
              <div style='color:#64748b; font-size:0.85em; margin-top:3px;'>
                {med.get("generic_name","")}
              </div>
            </div>
            <span style='background:rgba(0,212,170,0.1); color:#00d4aa; padding:5px 14px;
                         border-radius:20px; font-size:0.82em; font-weight:600; white-space:nowrap;'>
              💊 {med.get("plain_name","")}
            </span>
          </div>

          <!-- What it is + Why prescribed -->
          <div style='display:grid; grid-template-columns:1fr 1fr; gap:12px; margin-bottom:16px;'>
            <div style='background:rgba(0,212,170,0.05); border:1px solid rgba(0,212,170,0.12);
                        border-radius:12px; padding:14px;'>
              <div style='font-size:0.72em; letter-spacing:1.5px; text-transform:uppercase;
                          color:#00d4aa; font-weight:600; margin-bottom:8px;'>What is it?</div>
              <p style='margin:0; color:#cbd5e1; line-height:1.65; font-size:0.9em;'>
                {med.get("what_it_is","")}
              </p>
            </div>
            <div style='background:rgba(0,153,255,0.05); border:1px solid rgba(0,153,255,0.12);
                        border-radius:12px; padding:14px;'>
              <div style='font-size:0.72em; letter-spacing:1.5px; text-transform:uppercase;
                          color:#0099ff; font-weight:600; margin-bottom:8px;'>Why prescribed?</div>
              <p style='margin:0; color:#cbd5e1; line-height:1.65; font-size:0.9em;'>
                {med.get("why_prescribed","")}
              </p>
            </div>
          </div>

          <!-- How to take -->
          <div style='background:rgba(255,255,255,0.03); border-radius:12px; 
                      padding:14px; margin-bottom:16px; border-left:3px solid #00d4aa;'>
            <div style='font-size:0.72em; letter-spacing:1.5px; text-transform:uppercase;
                        color:#00d4aa; font-weight:600; margin-bottom:8px;'>📋 How to take it</div>
            <p style='margin:0; color:#cbd5e1; line-height:1.7; font-size:0.9em;'>
              {med.get("how_to_take","")}
            </p>
          </div>

          <!-- Side effects + Avoid -->
          <div style='display:grid; grid-template-columns:1fr 1fr; gap:12px; margin-bottom:16px;'>
            <div>
              <div style='font-size:0.72em; letter-spacing:1.5px; text-transform:uppercase;
                          color:#ffaa00; font-weight:600; margin-bottom:8px;'>⚡ Common Side Effects</div>
              <div style='line-height:2;'>{side_effects if side_effects else 
                "<span style='color:#475569; font-size:0.9em;'>None commonly reported</span>"}</div>
            </div>
            <div>
              <div style='font-size:0.72em; letter-spacing:1.5px; text-transform:uppercase;
                          color:#ff4444; font-weight:600; margin-bottom:8px;'>🚫 What to Avoid</div>
              <ul style='margin:0; padding-left:0; list-style:none;'>
                {avoid_items if avoid_items else 
                 "<li style='color:#475569; font-size:0.9em;'>No specific restrictions</li>"}
              </ul>
            </div>
          </div>

          <!-- Tips -->
          {f'''<div style="background:rgba(0,212,170,0.05); border-radius:10px; padding:12px 14px; 
                          margin-bottom:16px;">
            <span style="color:#00d4aa; font-size:0.82em;">💡 <b>Tip:</b></span>
            <span style="color:#94a3b8; font-size:0.88em; margin-left:4px;">{med.get("tips","")}</span>
          </div>''' if med.get("tips") else ""}

          <!-- Alternatives -->
          {alts_html}

        </div>"""

    # General advice footer
    if general_advice:
        html += f"""
        <div style='background:rgba(0,212,170,0.06); border:1px solid rgba(0,212,170,0.15);
                    border-radius:14px; padding:16px 20px; margin-top:4px;'>
          <span style='color:#00d4aa; font-size:0.82em; font-weight:600;'>📌 General Advice: </span>
          <span style='color:#94a3b8; font-size:0.9em; line-height:1.6;'>{general_advice}</span>
        </div>"""

    html += "</div>"
    return html

In [20]:
def analyze_prescription(
    image,
    age, gender, weight,
    conditions, allergies, current_meds,
    progress=gr.Progress()
):
   # ── Loading placeholders ───────────────────────────────────────────────
    loading_spinner = """
    <div style='text-align:center; padding:40px; font-family:DM Sans,sans-serif; color:#475569;'>
      <div style='font-size:2em; margin-bottom:12px; animation:spin 1.5s linear infinite; 
                  display:inline-block;'>⏳</div>
      <p style='margin:0; font-size:0.9em;'>Analyzing medicines...</p>
      <p style='margin:6px 0 0; font-size:0.78em; color:#334155;'>
        This may take 20–40 seconds
      </p>
    </div>"""

    empty_placeholder = """
    <div style='text-align:center; padding:30px; color:#334155;
                font-family:DM Sans,sans-serif; font-size:0.9em;'>
      Waiting for analysis to complete...
    </div>"""

    if image is None:
        empty = "<div style='text-align:center;padding:40px;color:#334155;" \
                "font-family:DM Sans,sans-serif;'>Upload a prescription to get started.</div>"
        yield empty, empty, empty
        return

    patient_profile = {
        "age": int(age) if age else 0,
        "gender": gender,
        "weight_kg": int(weight) if weight else None,
        "conditions": [c.strip() for c in conditions.split(",") if c.strip()],
        "allergies":  [a.strip() for a in allergies.split(",")  if a.strip()],
        "current_medications": [
            {"name": m.strip(), "dosage": "", "frequency": ""}
            for m in current_meds.split(",") if m.strip()
        ]
    }

    pil_image = Image.fromarray(image).convert("RGB")

    #enhance image
    #pil_image = preprocess_prescription(pil_image)
    
    # ── Call 1: Vision — extract medicines from image ──────────────────────
    progress(0.1, desc="Reading prescription image...")
    # Show spinners in all three tabs immediately
    yield empty_placeholder, loading_spinner, loading_spinner
    
    extracted = extract_prescription(pil_image)
    if extracted.get("parse_error"):
        err = "<div style='color:#ff4444;padding:20px;'>❌ Could not read prescription. Try a clearer image.</div>"
        yield err, err, err
        return

    progress(0.35, desc="Medicines extracted — running analysis...")
    medicines_html = medicines_to_html(extracted)
    # Push medicines now, keep spinners on education + safety tabs
    yield medicines_html, loading_spinner, loading_spinner

    
    # ── Call 2: Text — education + safety in one shot ──────────────────────
    progress(0.55, desc="Analyzing medicines...")

    # Short-circuit safety if profile is empty — still run education
    skip_safety = validate_safety_inputs(extracted, patient_profile) 

    raw_analysis = analyze_medicines(extracted, patient_profile)
   
    education, safety = split_analysis(raw_analysis)

    # Override safety with pre-validated clear result if profile was empty
    if skip_safety:
        safety = skip_safety

    progress(0.9, desc="Preparing results...")
    education_html = education_to_html(education)
    safety_html    = safety_to_html(safety)

    progress(1.0, desc="Done!")
    yield medicines_html, education_html, safety_html

In [21]:
with gr.Blocks(css=custom_css, title="RxSafe") as app:

    gr.HTML("""
    <div class='rx-header'>
      <h1 class='rx-logo'>💊 RxSafe</h1>
      <p class='rx-tagline'>Prescription Safety Companion · Powered by MedGemma</p>
      <span class='rx-disclaimer'>⚠️ Decision-support tool only — always confirm with your doctor or pharmacist</span>
    </div>
    """)

    with gr.Row(equal_height=False):

        # ── Left Column — Inputs (unchanged) ─────────────────────────────
        with gr.Column(scale=1, min_width=340):

            gr.HTML("<div class='panel-title'>📄 Prescription Image</div>")
            image_input = gr.Image(
                label="", type="numpy",
                elem_classes=["upload-box"], height=220
            )

            gr.HTML("<hr class='section-divider'><div class='panel-title'>👤 Patient Profile</div>")

            example_btn = gr.Button(
                "📋 Load Example Patient Profile",
                variant="secondary", elem_classes=["btn-secondary"]
            )

            with gr.Row():
                age_input    = gr.Number(label="Age", value=45, precision=0, minimum=0, maximum=120)
                gender_input = gr.Dropdown(["Male", "Female", "Other"], label="Gender", value="Male")
                weight_input = gr.Number(label="Weight (kg)", value=70, precision=0)

            conditions_input = gr.Textbox(
                label="Medical Conditions",
                placeholder="e.g. Type 2 Diabetes, Hypertension", lines=2
            )
            allergies_input = gr.Textbox(
                label="Known Allergies",
                placeholder="e.g. Penicillin, Sulfa drugs", lines=2
            )
            current_meds_input = gr.Textbox(
                label="Current Medications",
                placeholder="e.g. Metformin 500mg, Amlodipine 5mg", lines=3
            )

            gr.HTML("<div style='height:8px;'></div>")
            analyze_btn = gr.Button(
                "🔍 Analyze Prescription",
                variant="primary", elem_classes=["btn-primary"]
            )

        # ── Right Column — Tabbed Outputs ────────────────────────────────
        with gr.Column(scale=1, min_width=340):

            gr.HTML("<div class='panel-title'>📊 Results</div>")

            with gr.Tabs(elem_id="output-tabs") as output_tabs:

                with gr.TabItem("💊 Medicines", id="tab-medicines"):
                    medicines_output = gr.HTML(
                        value="<div style='text-align:center;padding:40px;color:#334155;"
                              "font-family:DM Sans,sans-serif;font-size:0.9em;'>"
                              "Upload a prescription to begin.</div>",
                        elem_id="medicines-out"
                    )

                with gr.TabItem("📖 Flash Cards", id="tab-education"):
                    education_output = gr.HTML(
                        value="<div style='text-align:center;padding:40px;color:#334155;"
                              "font-family:DM Sans,sans-serif;font-size:0.9em;'>"
                              "Detailed medicine explanations will appear here.</div>",
                        elem_id="education-out"
                    )
                     
                with gr.TabItem("🛡️ Safety Check", id="tab-safety"):
                    safety_output = gr.HTML(
                        value="<div style='text-align:center;padding:40px;color:#334155;"
                              "font-family:DM Sans,sans-serif;font-size:0.9em;'>"
                              "Safety results will appear here.</div>",
                        elem_id="safety-out"
                    )

               

    # ── Event Handlers ────────────────────────────────────────────────────
    def load_example():
        return 68, "Female", 60, "Type 2 Diabetes, Hypertension", \
               "Penicillin", "Metformin 500mg twice daily, Amlodipine 5mg once daily"

    example_btn.click(
        fn=load_example,
        outputs=[age_input, gender_input, weight_input,
                 conditions_input, allergies_input, current_meds_input]
    )

    analyze_btn.click(
        fn=analyze_prescription,
        inputs=[image_input, age_input, gender_input, weight_input,
                conditions_input, allergies_input, current_meds_input],
        outputs=[medicines_output, education_output, safety_output]
    )

/tmp/ipykernel_55/869048777.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, title="RxSafe") as app:


In [22]:
# launch application
if _launch_app:
    gr.close_all()
    
    app.launch(
        share=True,
        debug=False,
        server_name="0.0.0.0",
        server_port=7860,       
    )

* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://f43392525fe4ef4568.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
